# YOLOE-26 — Open-Vocabulary Video Surveillance Demos

**[PyConSG-2026 Talk:](https://pycon.sg/)** *Stop training, start prompting: The future of dynamic video surveillance*

3 demos:

1. **Semantic (text) prompt**
2. **Complex concept (text) prompt**
3. **Visual prompt**

## Setup

In [ ]:
import torch

# Check for GPU availability
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Please go to Runtime > Change runtime type and select a GPU hardware accelerator (e.g., T4, L4) to proceed.")
else:
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")

### Install Python packages

In [ ]:
%pip install ultralytics opencv-python-headless ffmpeg-python -q

### Mount Google Drive
- This is where media data for this demo is stored.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Define functions

In [ ]:
import subprocess

def convert_video_to_streamable(input_path, output_path):
    """
    Converts a video to a web-optimized MP4 using GPU hardware acceleration (NVENC).
    """
    command = [
        "ffmpeg",
        "-y",
        "-hwaccel", "cuda",             # Enable CUDA hardware acceleration
        "-i", input_path,
        "-c:v", "h264_nvenc",           # Use NVIDIA H.264 GPU encoder
        "-preset", "fast",              # Encoding speed preset
        "-movflags", "faststart",       # Optimized for web streaming
        "-pix_fmt", "yuv420p",          # Wide compatibility format
        "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        output_path
    ]

    print(f"Converting {input_path} using GPU acceleration...")
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    print(f"Successfully converted to {output_path}")


### Load YOLOE-26 model

In [ ]:

from ultralytics import YOLOE

# Load model
model = YOLOE("yoloe-26l-seg.pt")

# move weights to GPU
model.to("cuda")


---

## Text-Prompted Detection

YOLOE-26 is open-vocabulary: pass any list of strings to `model.set_classes(...)` and inference finds those classes out of the box.

YOLOE-26 achieves this using RepRTA (Re-Parameterizable Region-Text Alignment). During deployment, the text prompt is mathematically 'folded' directly into the model's inference weights.

**Business Impact:** In traditional AI, adding a new class like 'yellow suitcase' costs thousands of dollars and weeks of data labeling. Here, we just type it in plain English, it requires zero additional computational overhead at the edge, and it works instantly without retraining.

---
## Demo 1 — Open-Vocabulary Search (Suitcase)

**The Concept: Zero-Shot Detection**
Standard detectors only find what they were trained on. YOLOE-26 uses semantic embeddings to find any object you can describe in text.

**Workflow:**
1. **Define Target:** We pass the string `"yellow suitcase"` to the model.
2. **Feature Mapping:** The model aligns the text embedding with visual features in the video.
3. **Dynamic Inference:** The model highlights the specific target without ever having seen a 'yellow suitcase' during its initial training.

In [ ]:
# play source video

from IPython.display import HTML
from base64 import b64encode

video_path="/content/drive/MyDrive/PyCon26/yellow-suitcase-14.mp4"

mp4 = open(video_path,"rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=1280 controls autoplay muted>
      <source src="{data_url}" type="video/mp4">
</video>
""")

In [ ]:

classes = ["yellow suitcase"]
model.set_classes(classes)

print(f"YOLOE-26 model ready for inference on GPU. Classes: {classes}")

In [ ]:
import os
import shutil
import cv2

# 1. Setup paths
output_dir = "/content/yolo_output"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

base_name = os.path.splitext(os.path.basename(video_path))[0]
raw_video_output = os.path.join(output_dir, f"{base_name}_raw.avi")

# 2. Run prediction and manually write video
print("Starting inference and saving output video...")
results = model.predict(
    source=video_path,
    stream=True,
    conf=0.8,
    verbose=True
)

video_writer = None

for r in results:
    annotated_frame = r.plot(
        masks=False,
        labels=True,
        conf=False,
        boxes=True
    )

    if video_writer is None:
        h, w = annotated_frame.shape[:2]
        fps = 30
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter(raw_video_output, fourcc, fps, (w, h))

    video_writer.write(annotated_frame)

if video_writer:
    video_writer.release()

# 3. Convert to streamable MP4 (Ensures browser compatibility)
if os.path.exists(raw_video_output):
    print(f"Found raw AVI output at: {raw_video_output}")
    convert_video_to_streamable(raw_video_output, "/content/output_streamable.mp4")
    print("Processing complete. You can now run the video player cell.")
else:
    print("Error: Video was not generated.")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open("output_streamable.mp4","rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=1280 controls autoplay loop mute>
      <source src="{data_url}" type="video/mp4">
</video>
""")

---
## Demo 2 — Complex Behavior Prompting (Interaction)

**The Concept: Beyond Simple Objects**
"Persons fighting" is a complex behavior, not a static object. Traditional systems struggle with context and often just see bounding boxes of people. YOLOE, however, understands verbs and interactions, acting like an automated security guard.

**Workflow:**
1. **Behavioral Prompt:** We update the class list to `["persons fighting"]`.
2. **Contextual Analysis:** The model looks for visual patterns and spatial relationships that match the semantic definition of the action.
3. **Real-time Alerting:** The system identifies the specific frame where the interaction begins, demonstrating instant re-tasking for safety and security.

In [ ]:

classes = ["persons fighting"]
model.set_classes(classes)

print(f"YOLOE-26 model ready for inference on GPU. Classes: {classes}")

In [ ]:
import os
import shutil
import cv2

video_path="/content/drive/MyDrive/PyCon26/street-fight.mp4"

# 1. Setup paths
output_dir = "/content/yolo_output"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

base_name = os.path.splitext(os.path.basename(video_path))[0]
raw_video_output = os.path.join(output_dir, f"{base_name}_raw.avi")

# 2. Run prediction and manually write video
print("Starting inference and saving output video...")
results = model.predict(
    source=video_path,
    stream=True,
    conf=0.35,
    verbose=True
)

video_writer = None

for r in results:
    annotated_frame = r.plot(
        masks=False,
        labels=True,
        conf=False,
        boxes=True
    )

    if video_writer is None:
        h, w = annotated_frame.shape[:2]
        fps = 30
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter(raw_video_output, fourcc, fps, (w, h))

    video_writer.write(annotated_frame)

if video_writer:
    video_writer.release()

# 3. Convert to streamable MP4
if os.path.exists(raw_video_output):
    print(f"Found raw AVI output at: {raw_video_output}")
    convert_video_to_streamable(raw_video_output, "/content/output_streamable.mp4")
    print("Processing complete. You can now run the video player cell.")
else:
    print("Error: Video was not generated.")

In [ ]:
# Play the input and output videos

from IPython.display import HTML
from base64 import b64encode


def get_video_tag(path, width=600):
    with open(path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return f'<video width="{width}" controls autoplay loop muted><source src="{data_url}" type="video/mp4"></video>'

# Load both input and output results
video1 = get_video_tag(video_path)
video4 = get_video_tag("/content/output_streamable.mp4")

HTML(f"""
<div style="display: flex; justify-content: space-around; align-items: flex-start;">
    <div style="text-align: center;"><h3>Input</h3>{video1}</div>
    <div style="text-align: center;"><h3>Output</h3>{video4}</div>
</div>
""")

---

## Demo 3 — Cross-Camera Search (Visual Prompting)

**The Concept: SAVPE (Segment Any Visual Prompt Embedding)**
Instead of searching for a word, you point at a specific instance. This is a massive time-saver. Instead of having security teams manually review 10 hours of video across 5 cameras to find a specific missing person, visual prompting automates the search across all feeds instantly.

**Workflow:**
1. **Auto-Segmentation:** Run a prompt-free pass on a reference frame to identify all distinct entities.
2. **Operator Selection:** The operator chooses a specific target mask (e.g., a specific individual).
3. **Visual Signature:** SAVPE extracts a unique 'visual fingerprint' from that mask.
4. **Multi-Stream Tracking:** The model searches for that exact signature across multiple different camera feeds (Cam 1 & Cam 4).

In [ ]:
from IPython.display import HTML
from base64 import b64encode

cam1_path="/content/drive/MyDrive/PyCon26/cam1.m4v"
cam4_path="/content/drive/MyDrive/PyCon26/cam4.m4v"

def get_video_tag(path, width=800):
    with open(path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    # Added 'loop' to the video attributes
    return f'<video width="{width}" controls autoplay muted><source src="{data_url}" type="video/mp4"></video>'

# Load both cam1 and cam4 results
video1 = get_video_tag(cam1_path)
video4 = get_video_tag(cam4_path)

HTML(f"""
<div style="display: flex; justify-content: space-around; align-items: flex-start;">
    <div style="text-align: center;"><h3>Cam 1</h3>{video1}</div>
    <div style="text-align: center;"><h3>Cam 4</h3>{video4}</div>
</div>
""")

In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

img_prompt_path="/content/drive/MyDrive/PyCon26/elderly_lady.png"

# Load the prompt image
img_prompt = cv2.imread(img_prompt_path)
h, w = img_prompt.shape[:2]

# Calculate a proportionate centered bbox (e.g., middle 60% width, 80% height)
x1, y1 = int(w * 0.35), int(h * 0.1)
x2, y2 = int(w * 0.6), int(h * 0.9)
selected_bbox = [x1, y1, x2, y2]

# Visualize the selection
preview = img_prompt.copy()
cv2.rectangle(preview, (selected_bbox[0], selected_bbox[1]), (selected_bbox[2], selected_bbox[3]), (0, 255, 0), 3)
print(f"Visual Prompt Selection (Size: {w}x{h}):")
cv2_imshow(preview)

# Create the visual prompt dictionary
visual_prompt = {
    "bboxes": np.array([selected_bbox], dtype=np.float32),
    "cls": np.array([0]),
}

### How YOLOE Visual Prompting Works

**The Core Concept: Dynamic Visual Search**
Instead of searching for a word (like "person"), YOLOE searches for specific pixels defined by your prompt.

*   **Region of Interest (ROI):** The bounding box defines which area of the image to look at.
*   **Feature Extraction:** The model processes that region to create a unique "visual signature" (embedding).
*   **Prototype Matching:** This signature becomes the search query. The model then scans video frames for regions with the highest visual similarity (cosine similarity).
*   **Masks vs. Boxes:** Using a precise mask is more effective than a box because it excludes background noise, leading to a cleaner "signature" and more stable tracking.

**Key Takeaway:** The `visual_prompt` dictionary isn't the data itself; it's the instruction for the model to build its own search query internally.

In [ ]:
import cv2
import os
import numpy as np
import glob
from google.colab.patches import cv2_imshow
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

# Define paths locally to ensure they are available
cam1_path = "/content/drive/MyDrive/PyCon26/cam1.m4v"
cam4_path = "/content/drive/MyDrive/PyCon26/cam4.m4v"

cctv_paths = [cam1_path, cam4_path]
output_folder = "/content/search_results"
os.makedirs(output_folder, exist_ok=True)

# Cleanup existing raw files
raw_to_delete = glob.glob(os.path.join(output_folder, "raw_cam*.avi"))
for f in raw_to_delete:
    try:
        os.remove(f)
    except:
        pass

for path in cctv_paths:
    base_name = os.path.basename(path).split('.')[0]
    # Updated extension to .avi for consistency
    raw_output_path = os.path.join(output_folder, f"raw_{base_name}.avi")
    streamable_output_path = os.path.join(output_folder, f"search_{base_name}.mp4")

    print(f"--- Performing Visual Search on: {path} ---")

    results = model.predict(
        source=path,
        visual_prompts=visual_prompt,
        refer_image=img_prompt,
        stream=True,
        predictor=YOLOEVPSegPredictor,
        verbose=True,
        conf=0.2
    )

    video_writer = None

    for result in results:
        if len(result.boxes) > 0:
            annotated_frame = result.plot(
                masks=False,
                labels=False,
                conf=False,
                boxes=True
            )

            if video_writer is None:
                h, w = annotated_frame.shape[:2]
                # Updated to XVID codec
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                video_writer = cv2.VideoWriter(raw_output_path, fourcc, 30, (w, h))

            video_writer.write(annotated_frame)

    if video_writer:
        video_writer.release()
        print(f"Raw search video saved to {raw_output_path}")
        try:
            convert_video_to_streamable(raw_output_path, streamable_output_path)
            print(f"Streamable version ready at: {streamable_output_path}")
        except Exception as e:
            print(f"Conversion failed for {base_name}: {e}")
    else:
        print(f"No matches found in {path}, no video created.")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def get_video_tag(path, width=600):
    with open(path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    # Added 'loop' to the video attributes
    return f'<video width="{width}" controls autoplay muted loop><source src="{data_url}" type="video/mp4"></video>'

# Load both cam1 and cam4 results
video1 = get_video_tag("/content/search_results/search_cam1.mp4")
video4 = get_video_tag("/content/search_results/search_cam4.mp4")

HTML(f"""
<div style="display: flex; justify-content: space-around; align-items: flex-start;">
    <div style="text-align: center;"><h3>Cam 1 Result</h3>{video1}</div>
    <div style="text-align: center;"><h3>Cam 4 Result</h3>{video4}</div>
</div>
""")

## Summary

The open-vocabulary capabilities of YOLOE-26 represent a major shift in how we deploy video AI:

*   **Zero Retraining Costs:** Add new detection targets on the fly without expensive data collection or model fine-tuning.
*   **Instant Adaptability:** Respond to dynamic security or business needs in seconds just by typing a prompt or clicking a target.
*   **Unified Vision-Language Understanding:** Move beyond rigid bounding boxes to tracking complex behaviors and unique visual signatures seamlessly across camera networks.